In [1]:
!pip install seaborn



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)

print("✅ Libraries imported!")
print(f"Pandas version: {pd.__version__}")

✅ Libraries imported!
Pandas version: 2.2.3


In [ ]:
# Load training patient IDs
train_patients = pd.read_csv('../../data/processed/train_patients.csv')
train_patient_ids = train_patients['SUBJECT_ID'].values

print(f"✅ Training patients loaded: {len(train_patient_ids):,}")
print(f"First 10 patient IDs: {train_patient_ids[:10]}")



In [ ]:
# Load lab items dictionary to understand what each ITEMID means
lab_items = pd.read_csv('../../data/raw/D_LABITEMS.csv')

print(f"✅ Loaded {len(lab_items):,} lab item definitions")
print("\nFirst 10 lab items:")
print(lab_items.head(10))

print("\nColumns:")
print(lab_items.columns.tolist())

In [7]:
# KEY LAB TESTS FOR ICU DIAGNOSIS
# Selected based on clinical importance for sepsis, pneumonia, AKI, etc.

important_labs = {
    # Hematology (Blood Cells)
    '51301': 'White Blood Cells (WBC)',
    '51300': 'WBC Count',
    '51221': 'Hematocrit',
    '51222': 'Hemoglobin',
    '51265': 'Platelet Count',
    '51277': 'Red Blood Cells',
    
    # Chemistry (Electrolytes & Metabolic)
    '50912': 'Creatinine',
    '50971': 'Potassium',
    '50983': 'Sodium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50960': 'Magnesium',
    
    # Kidney Function
    '51006': 'Blood Urea Nitrogen (BUN)',
    '50920': 'Estimated GFR',
    
    # Liver Function
    '50861': 'ALT',
    '50878': 'AST',
    '50885': 'Bilirubin (Total)',
    '50863': 'Alkaline Phosphatase',
    
    # Infection/Inflammation Markers
    '50889': 'C-Reactive Protein (CRP)',
    '51003': 'Troponin',
    
    # Coagulation
    '51237': 'INR',
    '51274': 'PT',
    '51275': 'PTT',
    
    # Arterial Blood Gas
    '50820': 'pH',
    '50821': 'pO2',
    '50818': 'pCO2',
    '50813': 'Lactate',
    
    # Additional
    '50868': 'Anion Gap',
    '50970': 'Phosphate',
}

print(f"🎯 Tracking {len(important_labs)} important lab tests")
print("\nLab tests being tracked:")
for itemid, name in important_labs.items():
    print(f"  {itemid}: {name}")

🎯 Tracking 31 important lab tests

Lab tests being tracked:
  51301: White Blood Cells (WBC)
  51300: WBC Count
  51221: Hematocrit
  51222: Hemoglobin
  51265: Platelet Count
  51277: Red Blood Cells
  50912: Creatinine
  50971: Potassium
  50983: Sodium
  50902: Chloride
  50882: Bicarbonate
  50931: Glucose
  50893: Calcium
  50960: Magnesium
  51006: Blood Urea Nitrogen (BUN)
  50920: Estimated GFR
  50861: ALT
  50878: AST
  50885: Bilirubin (Total)
  50863: Alkaline Phosphatase
  50889: C-Reactive Protein (CRP)
  51003: Troponin
  51237: INR
  51274: PT
  51275: PTT
  50820: pH
  50821: pO2
  50818: pCO2
  50813: Lactate
  50868: Anion Gap
  50970: Phosphate


In [8]:
print("Loading LABEVENTS.csv in chunks...")
print("This will take 10-30 minutes - BE PATIENT!")
print("=" * 60)

# Convert important lab codes to integers for filtering
important_lab_codes = [int(code) for code in important_labs.keys()]

# Load LABEVENTS in chunks and filter
chunk_size = 1_000_000  # 1 million rows at a time
lab_chunks = []

chunk_counter = 0
total_rows_processed = 0
rows_kept = 0

# Read file in chunks
for chunk in tqdm(pd.read_csv('../../data/raw/LABEVENTS.csv', chunksize=chunk_size)):
    chunk_counter += 1
    total_rows_processed += len(chunk)
    chunk_filtered = chunk[
        (chunk['SUBJECT_ID'].isin(train_patient_ids)) &
        (chunk['ITEMID'].isin(important_lab_codes))
    ]
    
    rows_kept += len(chunk_filtered)
    
    if len(chunk_filtered) > 0:
        lab_chunks.append(chunk_filtered)
    if chunk_counter % 5 == 0:
        print(f"  Processed {total_rows_processed:,} rows, kept {rows_kept:,} rows ({rows_kept/total_rows_processed*100:.2f}%)")

print("\nCombining chunks...")
labs_train = pd.concat(lab_chunks, ignore_index=True)

print("=" * 60)
print(f"✅ DONE!")
print(f"   Total rows processed: {total_rows_processed:,}")
print(f"   Rows kept: {len(labs_train):,}")
print(f"   Reduction: {(1 - len(labs_train)/total_rows_processed)*100:.1f}%")
print(f"   Unique patients: {labs_train['SUBJECT_ID'].nunique():,}")
print(f"   Unique lab types: {labs_train['ITEMID'].nunique():,}")


Loading LABEVENTS.csv in chunks...
This will take 10-30 minutes - BE PATIENT!


5it [00:03,  1.26it/s]

  Processed 5,000,000 rows, kept 1,761,272 rows (35.23%)


10it [00:08,  1.12it/s]

  Processed 10,000,000 rows, kept 3,508,650 rows (35.09%)


...

  Processed 20,000,000 rows, kept 7,095,657 rows (35.48%)


25it [00:20,  1.27it/s]

  Processed 25,000,000 rows, kept 9,104,528 rows (36.42%)


28it [00:23,  1.20it/s]



Combining chunks...
✅ DONE!
   Total rows processed: 27,854,055
   Rows kept: 10,229,663
   Reduction: 63.3%
   Unique patients: 21,794
   Unique lab types: 31


In [13]:
!pip install pyarrow


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# Save filtered labs as CSV (simpler, always works)
print("Saving filtered lab data as CSV...")
labs_train.to_csv('../../data/processed/labs_train_filtered.csv', index=False)
print(f"✅ Saved {len(labs_train):,} lab events to CSV")
print(f"   Takes about 30 seconds...")

import os
file_size = os.path.getsize('../../data/processed/labs_train_filtered.csv') / 1024**2
print(f"   File size: {file_size:.1f} MB on disk")

Saving filtered lab data as CSV...
✅ Saved 10,229,663 lab events to CSV
   Takes about 30 seconds...
   File size: 657.7 MB on disk


In [ ]:
print("🔍 Examining filtered lab data:")
print("=" * 60)

print(f"\nDataFrame shape: {labs_train.shape}")
print(f"\nColumns: {labs_train.columns.tolist()}")

print(f"\nFirst 10 rows:")
print(labs_train.head(10))

print(f"\nData types:")
print(labs_train.dtypes)

print(f"\n📊 Lab test distribution (top 20):")
lab_counts = labs_train['ITEMID'].value_counts()
for itemid, count in lab_counts.head(20).items():
    lab_name = important_labs.get(str(itemid), 'Unknown')
    print(f"   {itemid}: {lab_name:40} {count:,} measurements")

print(f"\n📊 Measurements per patient:")
measurements_per_patient = labs_train.groupby('SUBJECT_ID').size()
print(f"   Min: {measurements_per_patient.min()}")
print(f"   Max: {measurements_per_patient.max()}")
print(f"   Mean: {measurements_per_patient.mean():.1f}")
print(f"   Median: {measurements_per_patient.median():.1f}")

print(f"\n⚠️  Missing values:")
print(labs_train.isnull().sum())

In [17]:
print("🧹 Cleaning lab data...")
print("=" * 60)

# Focus on VALUENUM (numeric lab values)
# Count issues before cleaning
print(f"Missing VALUENUM: {labs_train['VALUENUM'].isnull().sum():,} ({labs_train['VALUENUM'].isnull().sum()/len(labs_train)*100:.2f}%)")

# Drop rows with missing VALUENUM
labs_clean = labs_train.dropna(subset=['VALUENUM']).copy()

print(f"\n✅ After removing missing values:")
print(f"   Rows: {len(labs_clean):,} (removed {len(labs_train) - len(labs_clean):,})")
print(f"   Patients: {labs_clean['SUBJECT_ID'].nunique():,}")

# Check for outliers (extremely high or low values)
print(f"\n📊 VALUENUM statistics:")
print(labs_clean['VALUENUM'].describe())

print("\n✅ Data cleaned!")

🧹 Cleaning lab data...
Missing VALUENUM: 56,400 (0.55%)

✅ After removing missing values:
   Rows: 10,173,263 (removed 56,400)
   Patients: 21,794

📊 VALUENUM statistics:
count    1.017326e+07
mean     5.055153e+01
std      1.291151e+02
min     -2.100000e+01
25%      6.700000e+00
50%      1.600000e+01
75%      5.900000e+01
max      3.640000e+04
Name: VALUENUM, dtype: float64

✅ Data cleaned!


In [18]:
print("📊 Aggregating labs per patient...")
print("This will take 5-10 minutes...")
print("=" * 60)

# For each patient, for each lab test, calculate: mean, min, max, std
patient_lab_features = []

# Get list of unique patients
unique_patients = labs_clean['SUBJECT_ID'].unique()
print(f"Processing {len(unique_patients):,} patients...")

# Process in batches for progress tracking
from tqdm import tqdm

for patient_id in tqdm(unique_patients):
    # Get all labs for this patient
    patient_labs = labs_clean[labs_clean['SUBJECT_ID'] == patient_id]
    
    # Initialize feature dict
    features = {'SUBJECT_ID': patient_id}
    
    # For each lab type
    for itemid in important_lab_codes:
        # Get values for this lab type
        lab_values = patient_labs[patient_labs['ITEMID'] == itemid]['VALUENUM']
        
        if len(lab_values) > 0:
            # Calculate statistics
            features[f'lab_{itemid}_mean'] = lab_values.mean()
            features[f'lab_{itemid}_min'] = lab_values.min()
            features[f'lab_{itemid}_max'] = lab_values.max()
            features[f'lab_{itemid}_std'] = lab_values.std() if len(lab_values) > 1 else 0
            features[f'lab_{itemid}_count'] = len(lab_values)
        else:
            # No measurements for this lab
            features[f'lab_{itemid}_mean'] = np.nan
            features[f'lab_{itemid}_min'] = np.nan
            features[f'lab_{itemid}_max'] = np.nan
            features[f'lab_{itemid}_std'] = np.nan
            features[f'lab_{itemid}_count'] = 0
    
    patient_lab_features.append(features)

# Create DataFrame
lab_features_df = pd.DataFrame(patient_lab_features)

print("\n" + "=" * 60)
print(f"✅ DONE!")
print(f"   Feature matrix shape: {lab_features_df.shape}")
print(f"   Patients: {len(lab_features_df):,}")
print(f"   Features per patient: {lab_features_df.shape[1] - 1}")

📊 Aggregating labs per patient...
This will take 5-10 minutes...
Processing 21,794 patients...


100%|███████████████████████████████████████████| 21794/21794 [06:37<00:00, 54.77it/s]



✅ DONE!
   Feature matrix shape: (21794, 156)
   Patients: 21,794
   Features per patient: 155


In [ ]:
print("🔍 Examining feature matrix:")
print("=" * 60)

print(f"\nShape: {lab_features_df.shape}")
print(f"Patients: {len(lab_features_df):,}")
print(f"Features: {lab_features_df.shape[1] - 1} (excluding SUBJECT_ID)")

print(f"\nFirst patient's features:")
print(lab_features_df.iloc[0])

print(f"\n📊 Missing value rates per feature (top 20):")
missing_rates = (lab_features_df.isnull().sum() / len(lab_features_df) * 100).sort_values(ascending=False)
print(missing_rates.head(20))

print(f"\n📊 Feature statistics (first 5 features):")
print(lab_features_df.describe().iloc[:, 1:6])  # Skip SUBJECT_ID, show first 5 features

In [20]:
# Save aggregated features
print("Saving lab feature matrix...")
lab_features_df.to_csv('../../data/processed/lab_features_train.csv', index=False)
print(f"✅ Saved lab features: {lab_features_df.shape}")
print(f"   File: data/processed/lab_features_train.csv")

# Check file size
import os
file_size = os.path.getsize('../../data/processed/lab_features_train.csv') / 1024**2
print(f"   File size: {file_size:.1f} MB")

Saving lab feature matrix...
✅ Saved lab features: (21794, 156)
   File: data/processed/lab_features_train.csv
   File size: 23.9 MB
